> PostgreSQL 14+ practice. Run `sql/00_sample_data.sql` first, then `SET search_path TO sql_practice, public;` in this connection. Run cells in order in the same SQL session. See README.md for setup and expected results. These are SQL cells; the original Python notebook metadata did not make them executable as Python.


## Creating a filtered view

### Business requirement: The finance department regularly analyzes successful transactions. Create a reusable dataset that excludes failed and pending transactions.

In [ ]:
-- Save a reusable successful-payment query; creating a view does not copy or change base rows.
CREATE OR REPLACE VIEW vv_successful_transaction AS
SELECT transaction_id, customer_id, amount, transaction_date
FROM payment_transactions
WHERE status = 'Success';

## Creating a view with JOINs

### Business requirement: Analysts need transaction records enriched with customer names and countries without repeatedly writing JOIN statements.

In [ ]:
-- Enrich payments with customer details while preserving unmatched payments.
CREATE OR REPLACE VIEW vv_customer_transaction AS
SELECT t.transaction_id, t.customer_id, t.amount, t.status, c.customer_name, c.country, t.transaction_date
FROM
    payment_transactions t
    LEFT JOIN customers c ON t.customer_id = c.customer_id;

## Views with CASE WHEN and conditional aggregation

### Business requirement: The operations department wants to monitor transaction success rates and payment statuses for each customer.

In [ ]:
-- Aggregate status counts and percentage per customer; NULLIF protects the denominator.
CREATE OR REPLACE VIEW vv_customer_payment_metrics AS
SELECT
    customer_id,
    COUNT(*) AS total_transaction,
    SUM(
        CASE
            WHEN status = 'Success' THEN 1
            ELSE 0
        END
    ) AS successful_transactions,
    SUM(
        CASE
            WHEN status = 'Failed' THEN 1
            ELSE 0
        END
    ) AS failed_transactions,
    SUM(
        CASE
            WHEN status = 'Pending' THEN 1
            ELSE 0
        END
    ) AS pending_transactions,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN status = 'Success' THEN 1
                ELSE 0
            END
        ) / NULLIF(COUNT(*), 0),
        2
    ) AS success_rate
FROM payment_transactions
GROUP BY
    customer_id;

## Creating a data-cleaning staging view

### Business requirement: Create a standardized customer dataset for downstream transformations.

In [ ]:
-- TRIM removes surrounding spaces, LOWER/UPPER standardize case, and NULLIF turns blanks into NULL.
CREATE OR REPLACE VIEW vv_customer_staging AS
SELECT
    customer_id,
    NULLIF(LOWER(TRIM(email)), '') AS email,
    NULLIF(UPPER(TRIM(country)), '') AS country,
    NULLIF(TRIM(customer_name), '') AS customer_name
FROM raw_customer;

## Creating a view with multiple CTEs

### Business requirement: Calculate the net amount of every successful transaction after deducting refunds, without duplicating transaction records.

In [ ]:
-- Aggregate refunds before joining to prevent payment duplication when multiple refunds exist.
CREATE OR REPLACE VIEW vv_net_transactions AS
WITH successful_transaction AS (
    SELECT transaction_id, customer_id, amount, transaction_date
    FROM payment_transactions WHERE status = 'Success'
), refunded_summary AS (
    SELECT transaction_id, SUM(refund_amount) AS total_refunded
    FROM refunds
    GROUP BY transaction_id
)
SELECT t.transaction_id, t.customer_id, t.transaction_date,
    t.amount AS original_amount,
    COALESCE(r.total_refunded, 0) AS refunded_amount,
    t.amount - COALESCE(r.total_refunded, 0) AS net_amount
FROM successful_transaction t
LEFT JOIN refunded_summary r ON t.transaction_id = r.transaction_id;